# $\color{#dda}{\text{Parte 1: El ecosistema de Hugging Face}}$

**Temas**
1. Las librerías del ecosistema: `transformers`, `datasets`, `tokenizers` (y `huggingface_hub`).
2. Instalación y configuración en CPU o GPU.
3. Pipelines: una tarea completa en una línea.
4. Anatomía de un modelo: config, tokenizer y pesos.

Todo corre en CPU con modelos chicos (familia DistilBERT y GPT-2), para que la demo funcione en cualquier laptop.

## $\color{#dda}{\text{1. Las librerías}}$

| Librería | Qué resuelve | Detalle clave |
|---|---|---|
| `huggingface_hub` | Descargar y subir archivos del Hub, caché local, autenticación | Todo `from_pretrained` pasa por aquí |
| `transformers` | Arquitecturas de modelos, clases `Auto*`, `pipeline`, `Trainer`, `generate` | Más de 300 arquitecturas con la misma interfaz |
| `tokenizers` | Tokenización rápida (núcleo en Rust) | Es el motor detrás de `AutoTokenizer` |
| `datasets` | Cargar y procesar datos | Guarda en Apache Arrow con memory mapping: no carga todo a RAM |
| `accelerate` | Mismo código en CPU, 1 GPU o varias | `Trainer` lo usa por debajo |

El **Hub** (huggingface.co) es el repositorio central: cada modelo, dataset o Space es un repo de git con archivos versionados.

In [1]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.insert(0, "..")  # para importar hfdemo desde notebooks/

from hfdemo.config import OUT, setup
device = setup()

In [2]:
import datasets, huggingface_hub, tokenizers, torch, transformers

for lib in (transformers, datasets, tokenizers, huggingface_hub, torch):
    print(f"{lib.__name__:16s} {lib.__version__}")

transformers     5.20.2
datasets         5.1.0
tokenizers       0.23.3
huggingface_hub  2.3.0
torch            2.14.1+cpu


## $\color{#dda}{\text{2. Instalación y configuración (CPU / GPU)}}$

```bash
# Entorno aislado
python -m venv .venv

# CPU (Windows y macOS: el wheel por default de PyTorch ya es de CPU)
pip install torch transformers datasets accelerate

# GPU NVIDIA: PyTorch con CUDA desde su índice propio
pip install torch --index-url https://download.pytorch.org/whl/cu126
pip install transformers datasets accelerate
```

Transformers no tiene "versión GPU": usa el dispositivo que PyTorch tenga disponible. Lo único que cambia es instalar PyTorch con CUDA y mover modelo y tensores al dispositivo.

`setup()` (en `hfdemo/config.py`) elige el dispositivo así: `cuda` si hay GPU NVIDIA, `mps` en Apple Silicon y si no, `cpu`.

In [3]:
print("Dispositivo:", device, "| hilos de CPU:", torch.get_num_threads())
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# Dónde se guardan los modelos descargados (se cambia con la variable HF_HOME)
print("Caché del Hub:", huggingface_hub.constants.HF_HUB_CACHE)

Dispositivo: cpu | hilos de CPU: 2
Caché del Hub: C:\Users\hecto\.cache\huggingface\hub


La primera vez que se pide un modelo se descarga a la caché; las siguientes se lee del disco. Con `HF_HUB_OFFLINE=1` se trabaja sin red usando solo lo que ya está en caché.

## $\color{#dda}{\text{3. Pipelines}}$

`pipeline(tarea)` junta tres pasos: **tokenizar → modelo → posprocesar**. Si no se indica modelo, elige uno por default para la tarea; en la práctica conviene fijarlo para que el resultado sea reproducible.

In [4]:
from transformers import pipeline

clasificador = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=device,
)
clasificador([
    "I love studying NLP with Hugging Face!",
    "This homework is taking forever and I hate it.",
    "The movie was okay, nothing special.",
])

[{'label': 'POSITIVE', 'score': 0.9996519088745117},
 {'label': 'NEGATIVE', 'score': 0.9978814721107483},
 {'label': 'NEGATIVE', 'score': 0.992511510848999}]

La tercera frase es neutra, pero el modelo la marca NEGATIVE con mucha confianza: fue entrenado en SST-2, que solo tiene dos clases. Un score alto no significa que el modelo "entienda" el caso; significa que no tiene otra opción.

In [5]:
rellenar = pipeline("fill-mask", model="distilbert-base-uncased", device=device)
for r in rellenar("Paris is the [MASK] of France.", top_k=3):
    print(f"{r['token_str']:>10s}  {r['score']:.3f}")

   capital  0.982
birthplace  0.003
northernmost  0.001


In [6]:
ner = pipeline("token-classification", model="dslim/distilbert-NER", aggregation_strategy="first", device=device)
for ent in ner("Clement Delangue founded Hugging Face in New York with Julien Chaumond."):
    print(f"{ent['entity_group']:4s} {ent['word']:18s} {ent['score']:.3f}")

PER  Clement Delangue   0.999
ORG  Hugging Face       0.938
LOC  New York           0.966
PER  Julien Chaumond    0.998


In [7]:
generador = pipeline("text-generation", model="distilgpt2", device=device)
generador("Machine learning is", max_new_tokens=20, do_sample=False)[0]["generated_text"]

'Machine learning is a new way to learn about the world.\n\n\n\n\n\n\n\n\n\n\n'

La decodificación greedy termina repitiendo saltos de línea. Es un problema clásico de la estrategia de decodificación, no del pipeline: la Parte 5 lo explica y lo corrige.

Otras tareas con la misma interfaz: `zero-shot-classification`, `feature-extraction`, `automatic-speech-recognition`, `image-classification`, `image-text-to-text`, etc. Cambia la tarea y el modelo, no el código.

**Ojo con tutoriales viejos:** Transformers 5 quitó los pipelines `question-answering`, `summarization`, `translation` y `text2text-generation`. La recomendación oficial es resolverlos con `text-generation` y un modelo de chat (Parte 5). Las clases de modelo (`AutoModelForQuestionAnswering`, etc.) siguen existiendo.

### $\color{#dda}{\text{Qué hace un pipeline por dentro}}$
Repetimos el análisis de sentimiento a mano, paso por paso.

In [8]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

ckpt = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(ckpt)
model = AutoModelForSequenceClassification.from_pretrained(ckpt).to(device).eval()

# 1) Tokenizar: texto -> ids
entrada = tokenizer(["I love studying NLP with Hugging Face!"], return_tensors="pt").to(device)
print("input_ids:", entrada["input_ids"].tolist())

# 2) Modelo: ids -> logits
with torch.no_grad():
    logits = model(**entrada).logits
print("logits:", logits.tolist())

# 3) Posprocesar: logits -> probabilidades -> etiqueta
probs = logits.softmax(dim=-1)[0]
pred = probs.argmax().item()
print("etiqueta:", model.config.id2label[pred], "| score:", round(probs[pred].item(), 4))

input_ids: [[101, 1045, 2293, 5702, 17953, 2361, 2007, 17662, 2227, 999, 102]]


logits: [[-3.818129301071167, 4.1446146965026855]]
etiqueta: POSITIVE | score: 0.9997


Mismo resultado que el pipeline. Saber esto importa cuando hace falta algo que el pipeline no expone (embeddings, lotes grandes, otra forma de posprocesar).

## $\color{#dda}{\text{4. Anatomía de un modelo de Hugging Face}}$

Un checkpoint en el Hub es un repo con tres piezas:

| Pieza | Archivos | Qué contiene |
|---|---|---|
| **Config** | `config.json` | Arquitectura e hiperparámetros: capas, dimensiones, etiquetas, tokens especiales |
| **Tokenizer** | `tokenizer.json`, `tokenizer_config.json`, `vocab.txt` | Vocabulario y reglas para pasar de texto a ids |
| **Modelo** | `model.safetensors` | Los pesos (tensores) |

Las clases `Auto*` leen `config.json`, ven el campo `model_type` y eligen la clase correcta.

In [9]:
from huggingface_hub import HfApi

info = HfApi().model_info(ckpt, files_metadata=True)
for f in info.siblings:
    tam = f"{f.size / 1e6:8.2f} MB" if f.size else ""
    print(f"{f.rfilename:28s} {tam}")

.gitattributes                   0.00 MB
README.md                        0.01 MB
config.json                      0.00 MB
map.jpeg                         0.08 MB
model.safetensors              267.83 MB
onnx/added_tokens.json           0.00 MB
onnx/config.json                 0.00 MB
onnx/model.onnx                267.96 MB
onnx/special_tokens_map.json     0.00 MB
onnx/tokenizer.json              0.71 MB
onnx/tokenizer_config.json       0.00 MB
onnx/vocab.txt                   0.23 MB
pytorch_model.bin              267.84 MB
rust_model.ot                  267.85 MB
tf_model.h5                    267.95 MB
tokenizer_config.json            0.00 MB
vocab.txt                        0.23 MB


El repo trae pesos en varios formatos históricos (`pytorch_model.bin`, `tf_model.h5`, `rust_model.ot`, `onnx/`). Transformers descarga solo `model.safetensors`: es el formato actual porque carga rápido y, a diferencia de `pickle`, no puede ejecutar código al abrirse.

### $\color{#dda}{\text{Config}}$

In [10]:
from transformers import AutoConfig

config = AutoConfig.from_pretrained(ckpt)
print(type(config).__name__)
for campo in ("model_type", "n_layers", "n_heads", "dim", "hidden_dim", "vocab_size",
              "max_position_embeddings", "id2label", "pad_token_id"):
    print(f"{campo:24s} {getattr(config, campo)}")

DistilBertConfig
model_type               distilbert
n_layers                 6
n_heads                  12
dim                      768
hidden_dim               3072
vocab_size               30522
max_position_embeddings  512
id2label                 {0: 'NEGATIVE', 1: 'POSITIVE'}
pad_token_id             0


### $\color{#dda}{\text{Tokenizer}}$

In [11]:
print(type(tokenizer).__name__, "| vocabulario:", tokenizer.vocab_size)
print("Tokens especiales:", tokenizer.special_tokens_map)
texto = "Tokenization splits unbelievable words."
print(tokenizer.tokenize(texto))
print(tokenizer(texto)["input_ids"])

BertTokenizer | vocabulario: 30522
Tokens especiales: {'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}
['token', '##ization', 'splits', 'unbelievable', 'words', '.']
[101, 19204, 3989, 19584, 23653, 2616, 1012, 102]


### $\color{#dda}{\text{Modelo}}$

In [12]:
n_params = sum(p.numel() for p in model.parameters())
print(f"{type(model).__name__}: {n_params / 1e6:.1f} M parámetros")
print(f"Memoria: {model.get_memory_footprint() / 1e6:.0f} MB")
for nombre, modulo in model.named_children():
    print(f"  {nombre:14s} {type(modulo).__name__}")

DistilBertForSequenceClassification: 67.0 M parámetros
Memoria: 268 MB
  distilbert     DistilBertModel
  pre_classifier Linear
  classifier     Linear
  dropout        Dropout


El modelo se divide en el **cuerpo** (`distilbert`: embeddings + 6 capas transformer) y la **cabeza** (`pre_classifier` + `classifier`), que convierte el vector del token `[CLS]` en 2 logits. Esta división se retoma en la Parte 3.

### $\color{#dda}{\text{Guardar y volver a cargar}}$
`save_pretrained` escribe exactamente las mismas piezas que hay en el Hub.

In [13]:
destino = OUT / "anatomia"
model.save_pretrained(destino)
tokenizer.save_pretrained(destino)
for f in sorted(destino.iterdir()):
    print(f"{f.name:28s} {f.stat().st_size / 1e6:8.2f} MB")

recargado = pipeline("sentiment-analysis", model=str(destino), device=device)
recargado("Saved and reloaded without problems.")

config.json                      0.00 MB
model.safetensors              267.83 MB
tokenizer.json                   0.71 MB
tokenizer_config.json            0.00 MB


[{'label': 'POSITIVE', 'score': 0.9973703622817993}]

## $\color{#dda}{\text{Resumen}}$

- `huggingface_hub` descarga y cachea; `tokenizers` y `datasets` preparan los datos; `transformers` define modelos, pipelines y entrenamiento.
- CPU o GPU depende de cómo se instaló PyTorch; el código solo cambia en el `device`.
- `pipeline` = tokenizer + modelo + posprocesamiento.
- Todo modelo es **config + tokenizer + pesos**, y `from_pretrained` / `save_pretrained` los mueven como unidad.